# Patient Wait-Time Analysis


## Project Overview

This notebook examines patient wait times across hospital departments, shifts, and triage levels. Understanding wait time drivers helps hospital administrators improve staffing, scheduling, and patient experience.


## Learning Objectives

- Analyze wait time distributions across departments, shifts, and triage levels
- Identify day-of-week patterns in patient flow
- Quantify the relationship between staff count and wait times
- Compute P95 wait times as a service level benchmark


## Business or Research Problem

Long wait times reduce patient satisfaction scores, increase left-without-being-seen rates, and can compromise clinical outcomes in time-sensitive cases. This analysis identifies where delays concentrate.


## Why This Analysis Matters

Healthcare systems increasingly use wait time data for accreditation, reimbursement adjustments, and competitive positioning. Data-driven insights enable targeted operational improvements.


## Dataset Overview

Synthetic dataset of 3000 patient visits. Each record captures visit ID, date, department, shift, patient type, triage level, wait minutes, consultation minutes, staff count, and day of week.


## Dataset Source and License Notes

This dataset is entirely synthetic, generated with NumPy and pandas using seed=42. No real patient data is used. Educational use only.


## Environment Setup


In [ ]:
import importlib, subprocess, sys
for pkg in ['numpy', 'pandas', 'matplotlib', 'seaborn', 'scipy']:
    try:
        m = importlib.import_module(pkg)
        print(f'{pkg}: {m.__version__}')
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)


## Configuration / Constants


In [ ]:
SEED = 42
N = 3000
np.random.seed(SEED)


## Dataset Download or Loading

We generate 3000 synthetic patient visit records.


In [ ]:
np.random.seed(SEED)
departments = ['Emergency', 'Cardiology', 'Orthopedics', 'Pediatrics', 'General Medicine']
shifts = ['Day', 'Evening', 'Night']
patient_types = ['Emergency', 'Elective', 'Walk-in']
days_of_week = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

dept = np.random.choice(departments, N)
shift = np.random.choice(shifts, N, p=[0.45, 0.35, 0.20])
ptype = np.random.choice(patient_types, N, p=[0.30, 0.40, 0.30])
triage = np.random.choice([1, 2, 3, 4, 5], N, p=[0.05, 0.15, 0.35, 0.30, 0.15])
staff = np.random.randint(3, 15, N)
dow = np.random.choice(days_of_week, N)
dates = pd.to_datetime('2022-01-01') + pd.to_timedelta(np.random.randint(0, 365, N), unit='D')

# Wait time influenced by triage level, shift and staff
base_wait = 60 - triage * 8 + np.where(shift == 'Night', 20, 0) + np.where(shift == 'Evening', 10, 0)
base_wait += np.where(dept == 'Emergency', 15, 0) + (15 - staff)
wait_minutes = np.clip(base_wait + np.random.normal(0, 15, N), 5, 180).astype(int)
consultation_minutes = np.clip(np.random.normal(20, 8, N), 5, 60).astype(int)

df = pd.DataFrame({
    'visit_id': [f'V{i:05d}' for i in range(N)],
    'date': dates,
    'department': dept,
    'shift': shift,
    'patient_type': ptype,
    'triage_level': triage,
    'wait_minutes': wait_minutes,
    'consultation_minutes': consultation_minutes,
    'staff_count': staff,
    'day_of_week': dow
})
print(df.shape)
df.head()


## Data Validation Checks


In [ ]:
print('Shape:', df.shape)
print('Missing values:')
print(df.isnull().sum())
print('Wait minutes range:', df['wait_minutes'].min(), '-', df['wait_minutes'].max())
print('Triage levels:', sorted(df['triage_level'].unique()))


## Data Cleaning


In [ ]:
assert df['wait_minutes'].min() >= 0
assert df['triage_level'].between(1, 5).all()
assert df['staff_count'].between(1, 20).all()
print('All checks passed.')


## Exploratory Data Analysis


In [ ]:
print(df[['wait_minutes', 'consultation_minutes', 'staff_count', 'triage_level']].describe())


## Univariate Analysis

The overall distribution of wait times shows how service capacity is meeting demand.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df['wait_minutes'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Wait Times')
axes[0].set_xlabel('Wait Minutes')
axes[0].set_ylabel('Count')
axes[0].axvline(df['wait_minutes'].median(), color='red', linestyle='--', label=f'Median: {df["wait_minutes"].median():.0f}')
axes[0].legend()

triage_counts = df['triage_level'].value_counts().sort_index()
axes[1].bar(triage_counts.index, triage_counts.values, color='tomato', edgecolor='white')
axes[1].set_title('Patient Count by Triage Level')
axes[1].set_xlabel('Triage Level')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()


The wait time distribution is approximately normal with a right skew from high-acuity cases. Triage level 3 is most common, representing typical urgent care volume.


## Bivariate / Multivariate Analysis

Department and shift are key operational variables. We compare mean wait times across these dimensions.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dept_wait = df.groupby('department')['wait_minutes'].mean().sort_values(ascending=False)
dept_wait.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Mean Wait Time by Department')
axes[0].set_ylabel('Mean Wait (min)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')

shift_wait = df.groupby('shift')['wait_minutes'].mean().reindex(['Day', 'Evening', 'Night'])
shift_wait.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='white')
axes[1].set_title('Mean Wait Time by Shift')
axes[1].set_ylabel('Mean Wait (min)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()


Emergency department shows the longest waits. Night shifts have elevated wait times due to reduced staffing.


## Feature-Specific Insights

Triage level directly reflects clinical priority. Lower triage numbers (more urgent) should have shorter waits.


In [ ]:
triage_wait = df.groupby('triage_level')['wait_minutes'].mean()
fig, ax = plt.subplots(figsize=(8, 5))
triage_wait.plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='white')
ax.set_title('Mean Wait Time by Triage Level (1=Most Urgent)')
ax.set_xlabel('Triage Level')
ax.set_ylabel('Mean Wait (min)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()


As expected, triage level 1 (most urgent) has the shortest wait while level 5 (least urgent) waits longest, confirming the priority queue logic.


## Statistical Checks

We quantify the correlation between staff count and wait time, and test whether it is statistically significant.


In [ ]:
corr_coeff, p_val = stats.pearsonr(df['staff_count'], df['wait_minutes'])
print(f'Pearson correlation (staff vs wait): {corr_coeff:.3f}, p-value: {p_val:.4f}')
if p_val < 0.05:
    print('Statistically significant negative correlation: more staff -> shorter waits')
else:
    print('No statistically significant correlation detected')


## Visual Analysis

Day-of-week patterns reveal whether weekends present different capacity challenges.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_wait = df.groupby('day_of_week')['wait_minutes'].mean().reindex(dow_order)
dow_wait.plot(kind='bar', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Mean Wait Time by Day of Week')
axes[0].set_ylabel('Mean Wait (min)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=40, ha='right')

# P95 wait time by department
p95 = df.groupby('department')['wait_minutes'].quantile(0.95).sort_values(ascending=False)
p95.plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('P95 Wait Time by Department')
axes[1].set_ylabel('P95 Wait (min)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()


P95 wait times highlight worst-case scenarios that affect patient satisfaction the most.


## Key Findings


In [ ]:
median_wait = df['wait_minutes'].median()
mean_wait = df['wait_minutes'].mean()
p95_overall = df['wait_minutes'].quantile(0.95)
longest_dept = dept_wait.idxmax()
longest_dept_wait = dept_wait.max()
longest_shift = shift_wait.idxmax()
peak_dow = dow_wait.idxmax()

print(f'Overall median wait time: {median_wait:.0f} minutes')
print(f'Overall mean wait time: {mean_wait:.1f} minutes')
print(f'P95 wait time across all visits: {p95_overall:.0f} minutes')
print(f'Department with longest mean wait: {longest_dept} ({longest_dept_wait:.1f} min)')
print(f'Shift with longest mean wait: {longest_shift}')
print(f'Day of week with longest mean wait: {peak_dow}')
print(f'Staff-wait correlation: {corr_coeff:.3f} (p={p_val:.4f})')


## Limitations

- Synthetic data oversimplifies real-world patient flow dynamics and case complexity.
- Staff count does not capture skill mix, specialization, or task overlap.
- Day-of-week patterns reflect random sampling rather than real demand cycles.
- Triage assignments are probabilistic, not clinically derived.


## Recommendations / Next Steps

- Focus staffing optimization on Emergency and Night shift, where waits are highest.
- Implement triage-based scheduling to fast-track level 1 and 2 patients.
- Monitor P95 wait times monthly to detect seasonal deterioration.
- Extend analysis with patient satisfaction scores to link wait times to experience outcomes.


## Common Mistakes

- Using mean instead of median or P95 for skewed wait time data.
- Ignoring triage level when comparing wait times across departments.
- Assuming correlation implies causality for staff count vs wait time.
- Forgetting to adjust for patient volume when comparing absolute wait times across shifts.


## Mini Challenge / Exercises

1. Compute the percentage of visits where wait time exceeds 60 minutes by department.
2. Plot a heatmap of mean wait time by department and shift.
3. Test whether Emergency patients have significantly longer waits than Elective patients.
4. Identify which triage level has the highest variability (std dev) in wait time.


## Final Summary / Key Takeaways

- Wait times vary substantially by department, shift, and triage level.
- Emergency department and Night shift are the primary bottlenecks.
- Higher staff counts are associated with shorter wait times, confirming resource-demand sensitivity.
- P95 benchmarks expose worst-case service gaps that mean averages would hide.
